Bu notebook, ham veriyi (raw data) makine öğrenmesi modelinin anlayabileceği ve öğrenebileceği bir formata dönüştürdüğümüz yerdir. Bir mühendis olarak burada yapacağımız her işlem, modelin başarısını doğrudan belirler.

1. Notebook Başlangıcı ve Veriyi Çağırma

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Veriyi yükleyelim
df = pd.read_csv("C:\\Users\\sibel\\source\\repos\\AquaPredict\\data\\water_potability.csv")
print("veri yüklendi.")

veri yüklendi.


2. Eksik Veri Yönetimi (Handling Missing Values) : EDA'da ph, Sulfate ve Trihalomethanes sütunlarında boşluklar olduğunu görmüştük.Makine öğrenmesi algoritmaları çoğunlukla boş verilerle çalışamaz. Bu boşlukları doldurmak zorundayız.En sağlıklı yöntemlerden biri, her sütunu kendi ortalaması (mean) ile doldurmaktır. Böylece veri setinin genel dağılımını bozmadan boşlukları kapatırız.(imputation)

In [11]:
# Sütun bazlı eksik verileri ortalama ile doldurma
df["ph"] = df["ph"].fillna(df["ph"].mean())
df["Sulfate"] = df["Sulfate"].fillna(df["Sulfate"].mean())
df["Trihalomethanes"] = df["Trihalomethanes"].fillna(df["Trihalomethanes"].mean())

# Kontrol: Boş veri kaldı mı?
print(df.isnull().sum())

ph                 0
Hardness           0
Solids             0
Chloramines        0
Sulfate            0
Conductivity       0
Organic_carbon     0
Trihalomethanes    0
Turbidity          0
Potability         0
dtype: int64


3. Train-Test Split : Modelin overfitting  olduğunu anlamak için eğitim verisi ile test verisini ayırırız.Modeli eğitim setiyle eğiteceğiz, ancak daha önce hiç görmediği "test seti" ile test edeceğiz.(cross-validation) Genelde %80 eğitim, %20 test olarak bölünür.

In [12]:
# Özellikler (X) ve Hedef (y) ayırımı
X = df.drop("Potability", axis=1) # Parametreler
y = df["Potability"]              # Hedef (İçilebilir mi?)

# Veriyi bölüyoruz
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"Eğitim Seti: {X_train.shape}")
print(f"Test Seti: {X_test.shape}")

Eğitim Seti: (2620, 9)
Test Seti: (656, 9)


4. Feature Scaling : Su verilerine bakarsak; ph değeri $7$ iken, Solids (Katı Maddeler) değeri $20.000$ olabilir. Model, büyük sayıları (20.000) küçük sayılardan (7) daha önemliymiş gibi algılayabilir.Bu da modeli yeterince doğru öğrenememesine neden olur. Tüm sayıları StandardScaler (Z-score normalizasyonu) kullanarak aynı ölçeğe ( $-3$ ile $+3$ arasına) çekerek modelin "adil" bir öğrenme yapmasını sağlıyoruz.

In [13]:
scaler = StandardScaler()

# Sadece eğitim verisiyle 'öğren' ve uygula (fit_transform)
X_train_scaled = scaler.fit_transform(X_train)

# Eğitimden öğrendiğin ölçeği test verisine sadece 'uygula' (transform)
X_test_scaled = scaler.transform(X_test)

5. İşlenmiş Veriyi Kaydetme : Bu notebook'un amacı veriyi hazırlamaktı. Şimdi bu tertemiz verileri bir sonraki modelleme notebook'una aktarmak için kaydediyoruz.

In [14]:
# Verileri bir üst dizindeki 'data' klasörüne kaydediyoruz
pd.DataFrame(X_train_scaled, columns=X.columns).to_csv("../data/data_X_train.csv", index=False)
pd.DataFrame(X_test_scaled, columns=X.columns).to_csv("../data/data_X_test.csv", index=False)
y_train.to_csv("../data/data_y_train.csv", index=False)
y_test.to_csv("../data/data_y_test.csv", index=False)

print("Veriler  'data' klasörüne başarıyla kaydedildi!")

Veriler  'data' klasörüne başarıyla kaydedildi!


6. StandardScaler : Veri setimizde bulunan pH (0-14) ve Solids (0-60.000) gibi parametreler çok farklı sayısal aralıklara sahiptir. StandardScaler, bu uçurumları ortadan kaldırarak tüm özellikleri benzer bir ölçeğe getirir; böylece modelin büyük sayılara (Solids) küçük sayılardan (pH) daha fazla önem vererek "yanlı" öğrenmesini engeller ve tüm parametrelerin karara adil katılmasını sağlar. Ayrıca, app.py üzerinden kullanıcıdan alacağımız ham verileri modelin anlayacağı "standart dile" tercüme etmek için bu dosya hayati bir köprü görevi görür.

In [15]:
import os
import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(scaler, "../models/scaler.pkl")
print("Scaler nesnesi '../models/scaler.pkl' yoluna başarıyla kaydedildi!")


Scaler nesnesi '../models/scaler.pkl' yoluna başarıyla kaydedildi!
